
# Bài tập chương 2 


In [1]:

import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Tạo bảng
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);
""")

# Dữ liệu gốc từ đề
students = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0),
]

courses = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc"),
]

cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)
cursor.executemany("INSERT INTO course VALUES (?, ?)", courses)
conn.commit()

print("Bảng student ban đầu:")
display(pd.read_sql_query("SELECT * FROM student", conn))

print("Bảng course ban đầu:")
display(pd.read_sql_query("SELECT * FROM course", conn))


Bảng student ban đầu:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


Bảng course ban đầu:


,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


## 1. Kết nối hai bảng

In [2]:

print("1.1. Tích Descartes")
df_cartesian = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_table_id, c.course_name
FROM student s
CROSS JOIN course c
ORDER BY s.student_id, c.id;
""", conn)
display(df_cartesian)
print("Tổng số dòng:", len(df_cartesian))


1.1. Tích Descartes


,student_id,name,class,course_id,score,course_table_id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


Tổng số dòng: 30


In [3]:

print("1.2. INNER JOIN")
df_inner = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
INNER JOIN course c ON s.course_id = c.id
ORDER BY s.student_id;
""", conn)
display(df_inner)

print("1.3. LEFT JOIN")
df_left = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
ORDER BY s.student_id;
""", conn)
display(df_left)

print("1.4. RIGHT JOIN (mô phỏng trong SQLite)")
df_right = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id AS course_table_id, c.course_name
FROM course c
LEFT JOIN student s ON s.course_id = c.id
ORDER BY c.id, s.student_id;
""", conn)
display(df_right)

print("1.5. FULL OUTER JOIN (mô phỏng trong SQLite)")
df_full = pd.read_sql_query("""
SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id AS course_table_id, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id

UNION ALL

SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.id AS course_table_id, c.course_name
FROM course c
LEFT JOIN student s ON s.course_id = c.id
WHERE s.student_id IS NULL
ORDER BY student_id;
""", conn)
display(df_full)


1.2. INNER JOIN


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


1.3. LEFT JOIN


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


1.4. RIGHT JOIN (mô phỏng trong SQLite)


,student_id,name,class,course_id,score,course_table_id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,None,None,NaN,NaN,26,Tin hoc
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
3,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke


1.5. FULL OUTER JOIN (mô phỏng trong SQLite)


,student_id,name,class,course_id,score,course_table_id,course_name
0,NaN,None,None,NaN,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None


## 2. Cập nhật `course_id` còn thiếu và loại bỏ dữ liệu không hợp lệ

In [4]:

# Cập nhật course_id còn thiếu bằng SQL
cursor.execute("UPDATE student SET course_id = 26 WHERE student_id = 3 AND course_id IS NULL;")
cursor.execute("UPDATE student SET course_id = 34 WHERE student_id = 9 AND course_id IS NULL;")
cursor.execute("UPDATE student SET course_id = 12 WHERE student_id = 10 AND course_id IS NULL;")
conn.commit()

print("Sau khi cập nhật course_id còn thiếu:")
display(pd.read_sql_query("SELECT * FROM student ORDER BY student_id", conn))


Sau khi cập nhật course_id còn thiếu:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,4,Le Thanh Huyen,Toan Tin,20,7.2
4,5,Vu Quoc Anh,May Tinh,24,8.0
5,6,Dang Thuy Linh,May Tinh,24,5.5
6,7,Bui Tien Dung,Kinh Te,34,9.2
7,8,Ho Ngoc Mai,Toan Tin,20,8.8
8,9,Duong Huu Phuc,Kinh Te,34,7.2
9,10,Cao Thi Hanh,May Tinh,12,7.0


In [5]:

# Loại bỏ bản ghi có course_id không tồn tại trong bảng course
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course);
""")
conn.commit()

print("Sau khi loại bỏ các bản ghi có course_id không tồn tại:")
display(pd.read_sql_query("SELECT * FROM student ORDER BY student_id", conn))


Sau khi loại bỏ các bản ghi có course_id không tồn tại:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,34,7.2
5,10,Cao Thi Hanh,May Tinh,12,7.0


In [6]:

print("2.a. Tổng số sinh viên, điểm trung bình của từng lớp")
df_class = pd.read_sql_query("""
SELECT class,
       COUNT(*) AS total_students,
       ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class
ORDER BY class;
""", conn)
display(df_class)

print("2.b. Tổng số sinh viên, điểm trung bình của từng môn học")
df_course = pd.read_sql_query("""
SELECT c.id AS course_id,
       c.course_name,
       COUNT(s.student_id) AS total_students,
       ROUND(AVG(s.score), 2) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY c.id;
""", conn)
display(df_course)

print("2.c. Phân loại thi đua theo điểm trung bình từng môn học")
df_classification = pd.read_sql_query("""
SELECT c.id AS course_id,
       c.course_name,
       ROUND(AVG(s.score), 2) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
           WHEN AVG(s.score) BETWEEN 6.0 AND 8.9 THEN 'Tot'
           ELSE 'Kem'
       END AS classification
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY c.id;
""", conn)
display(df_classification)


2.a. Tổng số sinh viên, điểm trung bình của từng lớp


,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


2.b. Tổng số sinh viên, điểm trung bình của từng môn học


,course_id,course_name,total_students,avg_score
0,12,Giai tich,2,6.85
1,26,Tin hoc,1,7.90
2,34,Thong ke,3,8.53


2.c. Phân loại thi đua theo điểm trung bình từng môn học


,course_id,course_name,avg_score,classification
0,12,Giai tich,6.85,Tot
1,26,Tin hoc,7.90,Tot
2,34,Thong ke,8.53,Tot


## 3. Xếp hạng sinh viên

In [7]:

print("3.a. Xếp hạng theo điểm số toàn bộ")
df_rank_all = pd.read_sql_query("""
SELECT student_id, name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) AS rank_desc,
       RANK() OVER (ORDER BY score ASC) AS rank_asc
FROM student
ORDER BY score DESC, student_id;
""", conn)
display(df_rank_all)

print("Top 3 cao nhất toàn bộ:")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score DESC) AS rnk
    FROM student
)
WHERE rnk <= 3
ORDER BY rnk, student_id;
""", conn))

print("Top 3 thấp nhất toàn bộ:")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (ORDER BY score ASC) AS rnk
    FROM student
)
WHERE rnk <= 3
ORDER BY rnk, student_id;
""", conn))


3.a. Xếp hạng theo điểm số toàn bộ


,student_id,name,class,course_id,score,rank_desc,rank_asc
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,5
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,5
2,3,Pham Van Nam,Toan Tin,26,7.9,3,4
3,9,Duong Huu Phuc,Kinh Te,34,7.2,4,3
4,10,Cao Thi Hanh,May Tinh,12,7.0,5,2
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6,1


Top 3 cao nhất toàn bộ:


,student_id,name,class,course_id,score,rnk
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


Top 3 thấp nhất toàn bộ:


,student_id,name,class,course_id,score,rnk
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,7.0,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3


In [8]:

print("3.b. Xếp hạng theo điểm số trong từng lớp")
df_rank_class = pd.read_sql_query("""
SELECT student_id, name, class, course_id, score,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_desc_in_class,
       RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_asc_in_class
FROM student
ORDER BY class, score DESC, student_id;
""", conn)
display(df_rank_class)

print("Top 3 cao nhất theo từng lớp:")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rnk
    FROM student
)
WHERE rnk <= 3
ORDER BY class, rnk, student_id;
""", conn))

print("Top 3 thấp nhất theo từng lớp:")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rnk
    FROM student
)
WHERE rnk <= 3
ORDER BY class, rnk, student_id;
""", conn))


3.b. Xếp hạng theo điểm số trong từng lớp


,student_id,name,class,course_id,score,rank_desc_in_class,rank_asc_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,2
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3,1
3,10,Cao Thi Hanh,May Tinh,12,7.0,1,2
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2,1
5,3,Pham Van Nam,Toan Tin,26,7.9,1,1


Top 3 cao nhất theo từng lớp:


,student_id,name,class,course_id,score,rnk
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,12,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


Top 3 thấp nhất theo từng lớp:


,student_id,name,class,course_id,score,rnk
0,9,Duong Huu Phuc,Kinh Te,34,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,2
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
4,10,Cao Thi Hanh,May Tinh,12,7.0,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


In [9]:

print("3.c. Xếp hạng theo điểm số trong từng mã môn học")
df_rank_course = pd.read_sql_query("""
SELECT student_id, name, class, course_id, score,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_desc_in_course,
       RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS rank_asc_in_course
FROM student
ORDER BY course_id, score DESC, student_id;
""", conn)
display(df_rank_course)

print("Top 3 cao nhất theo từng mã môn học:")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rnk
    FROM student
)
WHERE rnk <= 3
ORDER BY course_id, rnk, student_id;
""", conn))

print("Top 3 thấp nhất theo từng mã môn học:")
display(pd.read_sql_query("""
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS rnk
    FROM student
)
WHERE rnk <= 3
ORDER BY course_id, rnk, student_id;
""", conn))


3.c. Xếp hạng theo điểm số trong từng mã môn học


,student_id,name,class,course_id,score,rank_desc_in_course,rank_asc_in_course
0,10,Cao Thi Hanh,May Tinh,12,7.0,1,2
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,1,1
3,2,Tran Thi Lan,Kinh Te,34,9.2,1,2
4,7,Bui Tien Dung,Kinh Te,34,9.2,1,2
5,9,Duong Huu Phuc,Kinh Te,34,7.2,3,1


Top 3 cao nhất theo từng mã môn học:


,student_id,name,class,course_id,score,rnk
0,10,Cao Thi Hanh,May Tinh,12,7.0,1
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
2,3,Pham Van Nam,Toan Tin,26,7.9,1
3,2,Tran Thi Lan,Kinh Te,34,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,7.2,3


Top 3 thấp nhất theo từng mã môn học:


,student_id,name,class,course_id,score,rnk
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,7.0,2
2,3,Pham Van Nam,Toan Tin,26,7.9,1
3,9,Duong Huu Phuc,Kinh Te,34,7.2,1
4,2,Tran Thi Lan,Kinh Te,34,9.2,2
5,7,Bui Tien Dung,Kinh Te,34,9.2,2


## 4. Bổ sung `graduation_date`

In [10]:

cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME;")
conn.commit()

cursor.execute("""
WITH ranked AS (
    SELECT student_id,
           RANK() OVER (ORDER BY score DESC) AS score_rank
    FROM student
)
UPDATE student
SET graduation_date = DATETIME(
    'now',
    '+' || (
        SELECT score_rank
        FROM ranked
        WHERE ranked.student_id = student.student_id
    ) || ' days'
);
""")
conn.commit()

df_final = pd.read_sql_query("SELECT * FROM student ORDER BY score DESC, student_id;", conn)
print("Bảng student sau khi thêm graduation_date:")
display(df_final)


Bảng student sau khi thêm graduation_date:


,student_id,name,class,course_id,score,graduation_date
0,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-04 13:44:01
1,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-04 13:44:01
2,3,Pham Van Nam,Toan Tin,26,7.9,2026-04-06 13:44:01
3,9,Duong Huu Phuc,Kinh Te,34,7.2,2026-04-07 13:44:01
4,10,Cao Thi Hanh,May Tinh,12,7.0,2026-04-08 13:44:01
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-09 13:44:01
